# 📚 Ingestion Pipeline Demo
**Project 6 — Relationship Manager Copilot | Phase 2: Ingestion**

This notebook demonstrates the full knowledge ingestion pipeline:
1. Load documents from `data/raw/`
2. Chunk with structure-aware splitting
3. Embed with OpenAI `text-embedding-3-small`
4. Index into ChromaDB
5. Test a retrieval query with metadata filtering

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded.')

## Step 1: Load Documents

In [ ]:
from src.ingestion.loader import load_documents

docs = load_documents('../data/raw')
print(f'\nLoaded {len(docs)} documents')

# Show summary
for doc in docs:
    print(f'  [{doc.doc_type.value:8}] [{doc.sensitivity.value:10}] {doc.source} ({len(doc.content)} chars)')

## Step 2: Chunk Documents

In [ ]:
from src.ingestion.chunker import chunk_documents

chunks = chunk_documents(docs, chunk_size=800, chunk_overlap=200)
print(f'Total chunks: {len(chunks)}')

# Show first 3 chunks
print('\n--- Sample Chunks ---')
for chunk in chunks[:3]:
    print(f'\nChunk ID: {chunk.chunk_id}')
    print(f'Type: {chunk.doc_type.value} | Sensitivity: {chunk.sensitivity.value}')
    print(f'Section: {chunk.page_or_section}')
    print(f'Content preview: {chunk.content[:200]}...')

## Step 3: Embed Chunks (requires OPENAI_API_KEY)

In [ ]:
from src.ingestion.embedder import embed_chunks

chunks, embeddings = embed_chunks(chunks)
print(f'Embeddings: {len(embeddings)} vectors of dimension {len(embeddings[0])}')

## Step 4: Index into ChromaDB

In [ ]:
from src.ingestion.indexer import ChromaIndexer

chroma = ChromaIndexer(persist_dir='../data/chroma_db')
chroma.reset()  # Fresh start
chroma.index_chunks(chunks, embeddings)
print(f'Total in ChromaDB: {chroma.count()} chunks')

## Step 5: Test Retrieval with Metadata Filtering

In [ ]:
from src.ingestion.embedder import embed_query

# Query 1: Policy-only retrieval
query = 'What is the maximum equity allocation for a conservative client?'
query_vec = embed_query(query)

results = chroma.query(
    query_embedding=query_vec,
    top_k=3,
    doc_types=['policy'],
    sensitivity_max='premium',
)

print(f'Query: {query}')
print(f'Results: {len(results)}\n')
for r in results:
    print(f'Score: {r.score:.4f} | [{r.chunk.doc_type.value}] {r.chunk.source}')
    print(f'  Section: {r.chunk.page_or_section}')
    print(f'  Preview: {r.chunk.content[:200]}...')
    print()

In [ ]:
# Query 2: Access control — standard RM should NOT see restricted research
query2 = 'Emerging markets investment opportunities'
query_vec2 = embed_query(query2)

# Standard RM (access level 0 = public only)
results_standard = chroma.query(
    query_embedding=query_vec2,
    top_k=5,
    sensitivity_max='standard',
)

# Premium RM (access level 1 = public + internal)
results_premium = chroma.query(
    query_embedding=query_vec2,
    top_k=5,
    sensitivity_max='premium',
)

print(f'Standard RM sees {len(results_standard)} results')
print(f'Premium RM sees {len(results_premium)} results')

# Verify no restricted docs appear for standard RM
restricted_in_standard = [r for r in results_standard if r.chunk.sensitivity.value == 'restricted']
print(f'Restricted docs in standard results: {len(restricted_in_standard)} (should be 0)')